# Step 1d — PlanetScope peak-inundation check (catalogue search only)

Spec: `CLAUDE.md` §6 Step 1d, added 2026-09-20. **This is an addition and changes nothing.** The
Step 1c optical reference stays exactly as built from Sentinel-2 alone, so the Step 2 radar
comparison remains independent of anything here.

**Why.** Our Sentinel-2 record has no usable view of the valleys for 11–14 March 2023: no
acquisition at all on 11, 13 or 14 March, and the 12 March acquisition 0% clear over both
valleys. If PlanetScope has a clear view in that window it would be the only optical observation
of this flood near its peak, and it would test two things the current reference cannot:

1. whether the radar gate missed fields that were **visibly under water at the time**; and
2. whether **3 m resolution separates plastic mulch from standing water**, since at 10 m a
   Sentinel-2 pixel spans several raised beds.

**If no scene in 11–14 March is usable over the units, that is a finding, not a gap:** it would
mean even near-daily commercial 3 m imagery could not observe this event, which strengthens the
sensor-availability conclusion in exhibit §5.4.

**Search only.** This notebook performs no activation, no order and no download.

**Credential handling.** The Planet key is read from `PL_API_KEY` in the environment or the
gitignored `.env`. It is sent only as HTTP basic auth (key as username, empty password), is never
printed, never written into a cell or a saved response, and never placed in a URL.

**CHECKPOINT:** the scene table, the coverage of the units, and the access verdict. Stop here.

In [1]:
import hashlib
import json
import os
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import box, shape

import common as C

SEARCH_URL = "https://api.planet.com/data/v1/quick-search"
ITEM_TYPES = ["PSScene"]
AOI_LL = (-121.90, 36.60, -121.55, 36.98)      # valleys box, identical to Step 1c
PAJARO_LL = (-121.86, 36.84, -121.70, 36.95)   # lower Pajaro Valley, the breach area
START, END = "2023-03-10T00:00:00Z", "2023-03-21T00:00:00Z"   # 10-20 March inclusive
PEAK = ("2023-03-11", "2023-03-14")


def planet_key():
    '''Environment first, then the gitignored .env. Longest match wins: the .env carries a
    short placeholder alongside the real key (see SPEC_CHANGELOG 2026-09-20).'''
    best = (os.environ.get("PL_API_KEY") or "").strip()
    env = C.ROOT / ".env"
    if env.exists():
        for line in env.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line.startswith("PL_API_KEY") and "=" in line:
                v = line.split("=", 1)[1].strip().strip('"').strip("'")
                if len(v) > len(best):
                    best = v
    if not best:
        raise SystemExit("No PL_API_KEY in the environment or .env.")
    return best


KEY = planet_key()
print(f"Planet key loaded: sha256[:8]={hashlib.sha256(KEY.encode()).hexdigest()[:8]} "
      f"(value never printed; sent as basic auth only)")
print(f"n units = 1320 | date window {START[:10]} to 2023-03-20 | AOI {AOI_LL} | "
      f"item types {ITEM_TYPES} | endpoint {SEARCH_URL}")

Planet key loaded: sha256[:8]=152e3e31 (value never printed; sent as basic auth only)
n units = 1320 | date window 2023-03-10 to 2023-03-20 | AOI (-121.9, 36.6, -121.55, 36.98) | item types ['PSScene'] | endpoint https://api.planet.com/data/v1/quick-search


## The catalogue search

In [2]:
aoi_geom = json.loads(gpd.GeoSeries([box(*AOI_LL)]).to_json())["features"][0]["geometry"]
body = {"item_types": ITEM_TYPES, "filter": {"type": "AndFilter", "config": [
    {"type": "GeometryFilter", "field_name": "geometry", "config": aoi_geom},
    {"type": "DateRangeFilter", "field_name": "acquired",
     "config": {"gte": START, "lte": END}}]}}

feats, url, page = [], f"{SEARCH_URL}?_page_size=250", 0
while url and page < 40:
    r = (requests.post(url, json=body, auth=(KEY, ""), timeout=120) if page == 0
         else requests.get(url, auth=(KEY, ""), timeout=120))
    r.raise_for_status()
    j = r.json()
    feats += j.get("features", [])
    url, page = (j.get("_links") or {}).get("_next"), page + 1

units = gpd.read_file(C.DERIVED / "01_units_dwr.geojson")
units = units[units.is_unit.astype(bool)].to_crs("EPSG:4326")
units_m = units.to_crs(C.GRID_CRS)
unit_area = units_m.geometry.area.sum()
pajaro = box(*PAJARO_LL)

rows = []
for f in feats:
    p, g = f["properties"], shape(f["geometry"])
    hit = units.geometry.intersects(g)
    gm = gpd.GeoSeries([g], crs="EPSG:4326").to_crs(C.GRID_CRS).iloc[0]
    rows.append({
        "id": f["id"], "acquired_utc": p.get("acquired"), "date": (p.get("acquired") or "")[:10],
        "item_type": "PSScene", "satellite_id": p.get("satellite_id"),
        "instrument": p.get("instrument"), "quality_category": p.get("quality_category"),
        "cloud_percent_scene": p.get("cloud_percent"),
        "clear_percent_scene": p.get("clear_percent"),
        "visible_percent_scene": p.get("visible_percent"),
        "shadow_percent": p.get("shadow_percent"),
        "heavy_haze_percent": p.get("heavy_haze_percent"),
        "usable_data": p.get("usable_data"),
        "units_intersected": int(hit.sum()),
        "pct_units_intersected": 100 * hit.mean(),
        "pct_unit_area_covered": 100 * units_m.geometry.intersection(gm).area.sum() / unit_area,
        "intersects_lower_pajaro": bool(g.intersects(pajaro)),
        "n_permissions": len(f.get("_permissions") or []),
    })
scenes = pd.DataFrame(rows).sort_values("acquired_utc").reset_index(drop=True)
scenes.to_csv(C.DERIVED / "01d_planet_search.csv", index=False)

print(f"PSScene items returned: {len(scenes)}; touching at least one unit: "
      f"{int((scenes.units_intersected > 0).sum())}")
print(f"Wrote 01d_planet_search.csv")
print(f"n units = 1320 | date window 2023-03-10 to 2023-03-20 | AOI {AOI_LL} | "
      f"dataset: Planet Data API quick-search, item type PSScene")

PSScene items returned: 94; touching at least one unit: 78
Wrote 01d_planet_search.csv
n units = 1320 | date window 2023-03-10 to 2023-03-20 | AOI (-121.9, 36.6, -121.55, 36.98) | dataset: Planet Data API quick-search, item type PSScene


## Acquisitions by date

**`cloud_percent` here is scene-wide.** A PlanetScope scene is far larger than our fields, so
this is not the number that decides whether a usable observation exists over the units. The
clipped figure is discussed below and cannot be produced with the access this key has.

In [3]:
by_date = scenes.groupby("date").agg(
    scenes=("id", "size"),
    touching_units=("units_intersected", lambda s: int((s > 0).sum())),
    max_pct_units=("pct_units_intersected", "max"),
    max_pct_unit_area=("pct_unit_area_covered", "max"),
    best_scene_clear_pct=("clear_percent_scene", "max"),
    median_scene_cloud_pct=("cloud_percent_scene", "median"))
all_days = pd.date_range("2023-03-10", "2023-03-20").strftime("%Y-%m-%d")
by_date = by_date.reindex(all_days).fillna(0).astype({"scenes": int, "touching_units": int})
print("PlanetScope acquisitions over the valleys box, 10-20 March 2023:")
print(by_date.round(1).to_string())
missing = [d for d in all_days if by_date.loc[d, "scenes"] == 0]
print(f"\nDates with no PlanetScope acquisition at all: {missing or 'none'}")
print(f"\nFor comparison, Sentinel-2 (Step 1c) acquired on 12, 15, 17, 20, 22 and 25 March, and")
print("was 0% clear over both valleys on 12, 17 and 22 March.")
by_date.to_csv(C.DERIVED / "01d_planet_by_date.csv")
print("Wrote 01d_planet_by_date.csv")
print(f"n units = 1320 | dataset: Planet Data API quick-search, PSScene")

PlanetScope acquisitions over the valleys box, 10-20 March 2023:
            scenes  touching_units  max_pct_units  max_pct_unit_area  best_scene_clear_pct  median_scene_cloud_pct
2023-03-10       9               8           43.0               54.6                  22.0                    46.0
2023-03-11       7               7           70.6               55.0                  26.0                    88.0
2023-03-12       8               8           56.4               57.6                  38.0                    58.5
2023-03-13       8               7           76.4               64.0                  71.0                    41.0
2023-03-14       0               0            0.0                0.0                   0.0                     0.0
2023-03-15      10               8           55.0               56.9                  87.0                    47.5
2023-03-16       4               4           43.4               61.4                  97.0                     2.5
2023-03-17     

## The peak window, 11–14 March — scene by scene

In [4]:
peak = scenes[(scenes.date.between(*PEAK)) & (scenes.units_intersected > 0)]
cols = ["id", "acquired_utc", "satellite_id", "quality_category", "cloud_percent_scene",
        "clear_percent_scene", "visible_percent_scene", "units_intersected",
        "pct_unit_area_covered"]
print(f"PlanetScope scenes intersecting the units, {PEAK[0]} to {PEAK[1]}: {len(peak)}")
print(peak[cols].to_string(index=False, float_format=lambda v: f"{v:.1f}"))
print()
best = peak.sort_values(["clear_percent_scene", "pct_unit_area_covered"],
                        ascending=False).head(3)
print("Most promising scenes in the peak window, by scene-wide clear fraction:")
print(best[cols].to_string(index=False, float_format=lambda v: f"{v:.1f}"))
peak.to_csv(C.DERIVED / "01d_planet_peak_window.csv", index=False)
print("\nWrote 01d_planet_peak_window.csv")
print(f"n units = 1320 | date window {PEAK[0]} to {PEAK[1]} | dataset: Planet Data API "
      f"quick-search, PSScene")

PlanetScope scenes intersecting the units, 2023-03-11 to 2023-03-14: 22
                     id                acquired_utc satellite_id quality_category  cloud_percent_scene  clear_percent_scene  visible_percent_scene  units_intersected  pct_unit_area_covered
20230311_175302_69_2447 2023-03-11T17:53:02.690412Z         2447             test                   98                    1                      2                228                   12.8
20230311_175304_86_2447 2023-03-11T17:53:04.864814Z         2447             test                   99                    1                      1                932                   55.0
20230311_175307_03_2447 2023-03-11T17:53:07.039216Z         2447         standard                   88                   11                     11                386                   44.1
20230311_175309_21_2447 2023-03-11T17:53:09.213618Z         2447         standard                   71                   26                     29                 13       

## Access and quota: what this key can actually do

In [5]:
QUOTA = "https://api.planet.com/auth/v1/experimental/public/my/subscriptions"
RESV = "https://api.planet.com/account/v1/quota-reservations/"

subs = requests.get(QUOTA, auth=(KEY, ""), timeout=60)
resv = requests.get(RESV, auth=(KEY, ""), timeout=60)
print(f"subscriptions endpoint  HTTP {subs.status_code}, "
      f"{len(subs.json()) if subs.status_code == 200 else '?'} subscription(s)")
print(f"quota-reservations      HTTP {resv.status_code}, "
      f"{(resv.json().get('meta') or {}).get('count') if resv.status_code == 200 else '?'} "
      f"reservation(s)")

probe = scenes.sort_values("pct_unit_area_covered", ascending=False).head(5)
print("\nDownload permissions on the five scenes covering most unit area:")
for r in probe.itertuples():
    a = requests.get(
        f"https://api.planet.com/data/v1/item-types/PSScene/items/{r.id}/assets",
        auth=(KEY, ""), timeout=60)
    n = len(a.json()) if a.status_code == 200 else None
    print(f"  {r.id}  _permissions={r.n_permissions}  /assets HTTP {a.status_code} "
          f"-> {n} asset(s) available")

print()
print("VERDICT ON ACCESS")
print("  The key authenticates and searches the catalogue, but every scene returns an empty")
print("  _permissions list and zero available assets, and the account holds no subscription and")
print("  no quota reservation. This key has CATALOGUE SEARCH ACCESS ONLY.")
print("  No download can be sized or proposed, because none can be placed: the blocker is")
print("  entitlement, not quota. Planet's published Education & Research Basic terms (3,000 km2")
print("  per month, minimum 100 km2 charged per intersecting scene when clipping) are recorded")
print("  here as documentation, not as a measured property of this account.")
print(f"n units = 1320 | endpoints: {QUOTA}, {RESV} | dataset: Planet Data API")

subscriptions endpoint  HTTP 200, 0 subscription(s)
quota-reservations      HTTP 200, 0 reservation(s)

Download permissions on the five scenes covering most unit area:


  20230313_180045_08_24c7  _permissions=0  /assets HTTP 200 -> 0 asset(s) available


  20230316_183550_22_241c  _permissions=0  /assets HTTP 200 -> 0 asset(s) available


  20230318_183000_24_2492  _permissions=0  /assets HTTP 200 -> 0 asset(s) available


  20230318_175048_42_2459  _permissions=0  /assets HTTP 200 -> 0 asset(s) available


  20230312_175548_79_2421  _permissions=0  /assets HTTP 200 -> 0 asset(s) available

VERDICT ON ACCESS
  The key authenticates and searches the catalogue, but every scene returns an empty
  _permissions list and zero available assets, and the account holds no subscription and
  no quota reservation. This key has CATALOGUE SEARCH ACCESS ONLY.
  No download can be sized or proposed, because none can be placed: the blocker is
  entitlement, not quota. Planet's published Education & Research Basic terms (3,000 km2
  per month, minimum 100 km2 charged per intersecting scene when clipping) are recorded
  here as documentation, not as a measured property of this account.
n units = 1320 | endpoints: https://api.planet.com/auth/v1/experimental/public/my/subscriptions, https://api.planet.com/account/v1/quota-reservations/ | dataset: Planet Data API


## What the search can and cannot settle

**Cloud clipped to the unit footprint — the number that actually decides — is not available.**
Quick-search returns `cloud_percent`, `clear_percent` and `visible_percent` computed over the
whole scene, which is far larger than our 59.4 km² of strawberry fields. The only quantitative
route to a clipped figure is the `ortho_udm2` usable-data mask, which is an **asset download**,
and this account has no asset permissions.

There is one free qualitative route that was **not** taken here: every item advertises a
`thumbnail` link, which costs no quota. That would allow a visual check of whether the valleys
are under cloud in a given scene, but it is a preview rather than a measurement and no imagery
was fetched at this checkpoint.

In [6]:
print("SUMMARY FOR THE EXHIBIT")
n_peak = len(peak)
print(f"  PlanetScope DID acquire over the valleys on 11, 12 and 13 March 2023 — the window in")
print(f"  which Sentinel-2 had no usable view — with {n_peak} scenes intersecting the units.")
print(f"  It did NOT acquire on 14 March.")
print(f"  The best peak-window scene by scene-wide clear fraction is "
      f"{best.iloc[0]['id']} ({best.iloc[0]['acquired_utc'][:19]}Z), "
      f"{best.iloc[0]['clear_percent_scene']:.0f}% clear scene-wide, covering "
      f"{best.iloc[0]['pct_unit_area_covered']:.0f}% of unit area.")
print(f"  On 11 March, the morning after the breach, scene-wide cloud over the units runs "
      f"{peak[peak.date == '2023-03-11'].cloud_percent_scene.min():.0f}-"
      f"{peak[peak.date == '2023-03-11'].cloud_percent_scene.max():.0f}%.")
print()
print("  So the §5.4 conclusion needs stating precisely: it is not that no satellite could")
print("  observe this flood near its peak. Commercial near-daily 3 m imaging DID acquire on")
print("  11, 12 and 13 March. What the free public record lacked, a commercial constellation")
print("  had — subject to the same cloud, and behind an entitlement this project does not hold.")
print(f"n units = 1320 | date window 2023-03-10 to 2023-03-20 | dataset: Planet Data API "
      f"quick-search, PSScene")

SUMMARY FOR THE EXHIBIT
  PlanetScope DID acquire over the valleys on 11, 12 and 13 March 2023 — the window in
  which Sentinel-2 had no usable view — with 22 scenes intersecting the units.
  It did NOT acquire on 14 March.
  The best peak-window scene by scene-wide clear fraction is 20230313_175753_63_24d0 (2023-03-13T17:57:53Z), 71% clear scene-wide, covering 8% of unit area.
  On 11 March, the morning after the breach, scene-wide cloud over the units runs 71-99%.

  So the §5.4 conclusion needs stating precisely: it is not that no satellite could
  observe this flood near its peak. Commercial near-daily 3 m imaging DID acquire on
  11, 12 and 13 March. What the free public record lacked, a commercial constellation
  had — subject to the same cloud, and behind an entitlement this project does not hold.
n units = 1320 | date window 2023-03-10 to 2023-03-20 | dataset: Planet Data API quick-search, PSScene
